# Multi-Agent RAG with Langfuse Telemetry & Advanced Retrieval

Complete system featuring:
- **Multi-Agent Architecture**: Researcher, Analyzer, Writer agents
- **Advanced Retrieval**: HNSW + MMR + Dense+Sparse Reranking
- **Memory Checkpoints**: LangGraph persistence for multi-turn conversations
- **SQLite Database**: Permanent storage for documents and chat history
- **Langfuse Telemetry**: Full tracing and observability

## 0. Install Dependencies

In [2]:
# %pip install -U langchain langchain-core langgraph langchain-openai langchain-community langfuse
# %pip install -U hnswlib python-dotenv pydantic sqlalchemy chromadb bm25s rank-bm25
# %pip install -U langchain-text-splitters

## 1. Environment Setup & Langfuse Integration

In [1]:
import os
from dotenv import load_dotenv
from langfuse import get_client, observe

# Load environment variables
load_dotenv()

# Initialize Langfuse for tracing
# Set these in your .env file:
# LANGFUSE_PUBLIC_KEY=pk_...
# LANGFUSE_SECRET_KEY=sk_...
# LANGFUSE_HOST=https://cloud.langfuse.com

langfuse_client = get_client()

print("✓ Langfuse initialized for tracing")

✓ Langfuse initialized for tracing


## 2. SQLite Database Setup

In [7]:
import sqlite3
from datetime import datetime
import json

DB_PATH = "rag_system.db"

def init_database():
    """Initialize SQLite database with required tables."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Documents table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS documents (
        id TEXT PRIMARY KEY,
        title TEXT NOT NULL,
        content TEXT NOT NULL,
        metadata TEXT,
        embedding BLOB,
        sparse_vector TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    
    # Chat history table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS chat_history (
        id TEXT PRIMARY KEY,
        session_id TEXT NOT NULL,
        agent_name TEXT NOT NULL,
        role TEXT,
        content TEXT NOT NULL,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        trace_id TEXT
    )
    """)
    
    # Agent state checkpoints
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS agent_checkpoints (
        id TEXT PRIMARY KEY,
        session_id TEXT NOT NULL,
        agent_name TEXT NOT NULL,
        state TEXT NOT NULL,
        step_number INTEGER,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    
    conn.commit()
    conn.close()
    print("✓ SQLite database initialized at", DB_PATH)

init_database()

✓ SQLite database initialized at rag_system.db


## 3. Advanced Retrieval Setup (HNSW + MMR + Reranking)

In [ ]:
import numpy as np
from langchain_openai import OpenAIEmbeddings
import hnswlib
from typing import List
import uuid

# Initialize embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

class AdvancedRetriever:
    """Retriever with HNSW + MMR + Dense+Sparse Reranking."""

    def __init__(self, embedding_dim: int = 1536):
        self.embedding_dim = embedding_dim
        self.hnsw_index = hnswlib.Index(space='cosine', dim=embedding_dim)
        self.hnsw_index.init_index(max_elements=10000, ef_construction=100, M=4)
        self.documents = {}
        self.doc_embeddings = {}  # Cache embeddings by doc index
        self.next_id = 0

    @observe()
    def add_documents(self, docs: List[dict]):
        """Add documents with embeddings to HNSW index."""
        for doc in docs:
            doc_id = str(uuid.uuid4())
            embedding = np.array(embeddings.embed_query(doc["content"]), dtype=np.float32)
            self.hnsw_index.add_items(embedding.reshape(1, -1), [self.next_id])
            self.documents[self.next_id] = {"id": doc_id, **doc}
            self.doc_embeddings[self.next_id] = embedding  # Cache on add
            self.next_id += 1

    @observe()
    def retrieve_hnsw(self, query: str, k: int = 10) -> List[dict]:
        """Retrieve using HNSW (Hierarchical Navigable Small World)."""
        query_embedding = np.array(embeddings.embed_query(query), dtype=np.float32)
        self.hnsw_index.set_ef(max(k * 2, 50))
        labels, distances = self.hnsw_index.knn_query(query_embedding.reshape(1, -1), k=min(k, len(self.documents)))
        results = [self.documents[idx] for idx in labels[0] if idx in self.documents]
        return results

    @observe()
    def retrieve_mmr(self, query: str, k: int = 5, lambda_mult: float = 0.5) -> List[dict]:
        """Maximal Marginal Relevance — uses cached embeddings, no extra API calls."""
        query_embedding = np.array(embeddings.embed_query(query), dtype=np.float32)

        self.hnsw_index.set_ef(max(min(20, len(self.documents)) * 2, 50))
        candidates_labels, _ = self.hnsw_index.knn_query(query_embedding.reshape(1, -1), k=min(20, len(self.documents)))

        # Map candidate HNSW indices to docs and their cached embeddings
        candidates = []
        for idx in candidates_labels[0]:
            if idx in self.documents:
                candidates.append((idx, self.documents[idx], self.doc_embeddings[idx]))

        selected_indices = []
        remaining = list(range(len(candidates)))

        while len(selected_indices) < k and remaining:
            best_idx = None
            best_score = -float('inf')

            for i in remaining:
                _, _, cand_emb = candidates[i]
                relevance = np.dot(query_embedding, cand_emb)

                if selected_indices:
                    selected_embs = np.array([candidates[s][2] for s in selected_indices])
                    diversity = np.min(selected_embs @ cand_emb)
                else:
                    diversity = 1.0

                score = lambda_mult * relevance - (1 - lambda_mult) * diversity
                if score > best_score:
                    best_score = score
                    best_idx = i

            if best_idx is not None:
                selected_indices.append(best_idx)
                remaining.remove(best_idx)

        return [candidates[i][1] for i in selected_indices]

print("✓ Advanced Retriever initialized with HNSW + MMR")

## 4. Sparse & Dense Reranking

In [ ]:
from rank_bm25 import BM25Okapi
import re

class DenseSpaceReranker:
    """Reranker combining dense (semantic) and sparse (lexical) signals."""

    def __init__(self):
        self.embeddings = embeddings
        self.bm25 = None
        self.corpus = []
        self.corpus_embeddings = []  # Cache embeddings at fit time

    def fit(self, documents: List[dict]):
        """Fit BM25 and pre-compute embeddings for corpus."""
        self.corpus = documents
        tokenized_corpus = [self._tokenize(doc["content"]) for doc in documents]
        self.bm25 = BM25Okapi(tokenized_corpus)
        # Pre-compute all corpus embeddings once
        self.corpus_embeddings = [
            np.array(self.embeddings.embed_query(doc["content"]), dtype=np.float32)
            for doc in documents
        ]

    def _tokenize(self, text: str) -> List[str]:
        return re.findall(r'\w+', text.lower())

    @observe()
    def rerank(self, query: str, candidates: List[dict], k: int = 5,
               dense_weight: float = 0.7, sparse_weight: float = 0.3) -> List[dict]:
        """Rerank using combined dense+sparse scores — uses cached corpus embeddings."""
        query_embedding = np.array(self.embeddings.embed_query(query), dtype=np.float32)
        query_tokens = self._tokenize(query)
        sparse_scores = self.bm25.get_scores(query_tokens) if self.bm25 else [0] * len(self.corpus)

        scores = []
        for doc in candidates:
            if doc in self.corpus:
                idx = self.corpus.index(doc)
                doc_embedding = self.corpus_embeddings[idx]
                sparse_score = sparse_scores[idx]
            else:
                doc_embedding = np.array(self.embeddings.embed_query(doc["content"]), dtype=np.float32)
                sparse_score = 0

            dense_score = np.dot(query_embedding, doc_embedding)
            combined_score = dense_weight * dense_score + sparse_weight * (sparse_score / 100.0)
            scores.append((doc, combined_score))

        scores.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in scores[:k]]

print("✓ Dense+Sparse Reranker initialized")

## 5. Multi-Agent Architecture with Memory Checkpoints

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END, START
from typing import Annotated, TypedDict, Literal
import operator

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

class AgentState(TypedDict):
    """Shared state for multi-agent system."""
    task: str
    context: str
    retrieved_docs: Annotated[list, operator.add]
    researcher_output: str
    analyzer_output: str
    writer_output: str
    session_id: str
    trace_id: str
    messages: Annotated[list, operator.add]

researcher_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research specialist. Summarize the provided context clearly and extract the key information relevant to the task."),
    ("user", "Task: {task}\n\nContext:\n{context}")
])

analyzer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an analysis expert. Critically analyze the research findings, identify patterns, strengths, and limitations."),
    ("user", "Task: {task}\n\nResearch findings:\n{researcher_output}")
])

writer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Produce a clear, well-structured final answer based on the analysis."),
    ("user", "Task: {task}\n\nAnalysis:\n{analyzer_output}")
])

@observe()
def researcher_node(state: AgentState) -> AgentState:
    chain = researcher_prompt | llm
    result = chain.invoke({"task": state["task"], "context": state["context"]})
    output = result.content
    return {"researcher_output": output, "messages": [{"role": "researcher", "content": output}]}

@observe()
def analyzer_node(state: AgentState) -> AgentState:
    chain = analyzer_prompt | llm
    result = chain.invoke({"task": state["task"], "researcher_output": state["researcher_output"]})
    output = result.content
    return {"analyzer_output": output, "messages": [{"role": "analyzer", "content": output}]}

@observe()
def writer_node(state: AgentState) -> AgentState:
    chain = writer_prompt | llm
    result = chain.invoke({"task": state["task"], "analyzer_output": state["analyzer_output"]})
    output = result.content
    return {"writer_output": output, "messages": [{"role": "writer", "content": output}]}

print("✓ Multi-agent nodes initialized with real LLM calls")

## 6. Build LangGraph with Checkpoints

In [27]:
from langgraph.checkpoint.memory import MemorySaver

# Create checkpoint saver for memory persistence
# Note: SqliteSaver is not available in current LangGraph version
# Using MemorySaver for in-memory checkpoints
# Checkpoints are manually saved to SQLite in TelemetryLogger
checkpoint_storage = MemorySaver()

# Build graph
builder = StateGraph(AgentState)
builder.add_node("researcher", researcher_node)
builder.add_node("analyzer", analyzer_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "researcher")
builder.add_edge("researcher", "analyzer")
builder.add_edge("analyzer", "writer")
builder.add_edge("writer", END)

# Compile with checkpoint storage for memory persistence
agent_graph = builder.compile(checkpointer=checkpoint_storage)

print("✓ LangGraph compiled with SQLite checkpoint storage")

✓ LangGraph compiled with SQLite checkpoint storage


## 7. Logging & Tracing Utilities with Langfuse

In [ ]:
import logging
from uuid import uuid4

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class TelemetryLogger:
    """Logs to SQLite; @observe() on each method handles Langfuse tracing."""

    def __init__(self, session_id: str):
        self.session_id = session_id

    @observe()
    def log_agent_message(self, agent_name: str, role: str, content: str):
        """Log agent message to SQLite. Langfuse span created by @observe()."""
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        msg_id = str(uuid4())
        cursor.execute("""
            INSERT INTO chat_history (id, session_id, agent_name, role, content)
            VALUES (?, ?, ?, ?, ?)
        """, (msg_id, self.session_id, agent_name, role, content))
        conn.commit()
        conn.close()
        logger.info(f"[{agent_name}] {role}: {content[:100]}...")

    def log_checkpoint(self, agent_name: str, step: int, state: dict):
        """Save agent state checkpoint to SQLite."""
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        checkpoint_id = str(uuid4())
        state_json = json.dumps(state, default=str)
        cursor.execute("""
            INSERT INTO agent_checkpoints (id, session_id, agent_name, state, step_number)
            VALUES (?, ?, ?, ?, ?)
        """, (checkpoint_id, self.session_id, agent_name, state_json, step))
        conn.commit()
        conn.close()
        logger.info(f"Checkpoint saved: {agent_name} step {step}")

print("✓ Telemetry logger initialized")

## 8. Sample Usage: Run Multi-Agent RAG

In [ ]:
# Create retriever, reranker, and telemetry
retriever = AdvancedRetriever()
reranker = DenseSpaceReranker()

# Sample documents
sample_docs = [
    {"title": "RAG Basics", "content": "Retrieval-Augmented Generation combines retrieval and generation models..."},
    {"title": "Vector DBs", "content": "Vector databases store embeddings for semantic search..."},
    {"title": "Multi-Agent", "content": "Multi-agent systems coordinate multiple specialized agents..."},
]

# Add documents to retriever (embeddings are cached here)
retriever.add_documents(sample_docs)
reranker.fit(sample_docs)

print("✓ Retriever and reranker ready")

## 9. Execute Multi-Agent Workflow

In [ ]:
@observe(name="multi-agent-rag-pipeline")
def run_rag_pipeline(task: str, session_id: str, telemetry):
    """Execute complete RAG pipeline with all agents and tracing."""
    langfuse_trace_id = langfuse_client.get_current_trace_id()

    # Set trace input visible in Langfuse dashboard
    langfuse_client.set_current_trace_io(input={"task": task, "session_id": session_id})

    retrieved = retriever.retrieve_hnsw(task, k=10)
    telemetry.log_agent_message("retriever", "system", f"Retrieved {len(retrieved)} documents")

    diverse_docs = retriever.retrieve_mmr(task, k=5, lambda_mult=0.5)
    telemetry.log_agent_message("retriever", "system", f"Applied MMR, selected {len(diverse_docs)} diverse results")

    reranked = reranker.rerank(task, retrieved, k=3, dense_weight=0.7, sparse_weight=0.3)
    telemetry.log_agent_message("reranker", "system", f"Reranked to top {len(reranked)} results")

    initial_state = {
        "task": task,
        "context": "\n".join([f"- {doc['title']}: {doc['content'][:150]}" for doc in reranked]),
        "retrieved_docs": reranked,
        "researcher_output": "",
        "analyzer_output": "",
        "writer_output": "",
        "session_id": session_id,
        "trace_id": langfuse_trace_id,
        "messages": []
    }

    result = agent_graph.invoke(
        initial_state,
        config={"configurable": {"thread_id": session_id}}
    )

    # Set final output on the trace
    langfuse_client.set_current_trace_io(output={"writer_output": result.get("writer_output")})

    telemetry.log_checkpoint("workflow", 1, result)
    return result, langfuse_trace_id


# Run pipeline 3 times with different tasks
tasks = [
    "Explain multi-agent RAG systems",
    "How does vector similarity search work?",
    "What are the benefits of reranking in retrieval systems?",
]

for i, task in enumerate(tasks, 1):
    session_id = str(uuid4())
    telemetry = TelemetryLogger(session_id)
    result, langfuse_trace_id = run_rag_pipeline(task, session_id, telemetry)
    langfuse_client.flush()

    print(f"\n{'='*60}")
    print(f"RUN {i}  : {task}")
    print(f"{'='*60}")
    print(f"Trace Name : multi-agent-rag-pipeline")
    print(f"Session ID : {session_id}")
    print(f"Trace ID   : {langfuse_trace_id}")
    print(f"Writer Output:\n{result.get('writer_output', 'N/A')}")

## 10. View Telemetry & Chat History

In [ ]:
@observe()
def view_session_history(session_id: str):
    """Retrieve and display chat history for a session."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    cursor.execute("""
        SELECT agent_name, role, content, timestamp FROM chat_history
        WHERE session_id = ? ORDER BY timestamp
    """, (session_id,))

    messages = cursor.fetchall()
    conn.close()

    print(f"\n📜 Session History: {session_id}\n")
    for msg in messages:
        print(f"[{msg['agent_name']}] {msg['role']}: {msg['content'][:100]}...")
        print(f"   Timestamp: {msg['timestamp']}\n")

@observe()
def view_checkpoints(session_id: str):
    """Retrieve and display checkpoints for a session."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    cursor.execute("""
        SELECT agent_name, step_number, timestamp FROM agent_checkpoints
        WHERE session_id = ? ORDER BY step_number
    """, (session_id,))

    checkpoints = cursor.fetchall()
    conn.close()

    print(f"\n🔖 Checkpoints: {session_id}\n")
    for cp in checkpoints:
        print(f"Step {cp['step_number']}: {cp['agent_name']} @ {cp['timestamp']}")

# View history and checkpoints
view_session_history(session_id)
view_checkpoints(session_id)

print("\n✓ Access Langfuse dashboard: https://cloud.langfuse.com")
print(f"✓ Search for trace ID: {langfuse_trace_id}")

## 11A. How to Access Your Traces in Langfuse

To view your traces using the Trace ID:

1. **Go to Langfuse Dashboard**: https://cloud.langfuse.com
2. **Login** with your Langfuse credentials (same as LANGFUSE_PUBLIC_KEY)
3. **Find Your Project**: Select the project from the dashboard
4. **Search for Trace**: 
   - Click on "Traces" in the left menu
   - Paste your trace ID in the search box (shown below as `telemetry.trace_id`)
   - Or look for traces in your session time window
5. **View Details**:
   - Click on the trace to expand it
   - You'll see the complete execution timeline
   - Each `@observe()` decorated function shows up as a span
   - View inputs, outputs, duration, and any errors

**Example: Using Your Trace ID**

## 11. Summary

This notebook demonstrates:

1. **Multi-Agent Architecture**: Researcher → Analyzer → Writer pipeline
2. **Advanced Retrieval**: 
   - HNSW for efficient similarity search
   - MMR for diverse results
   - Dense+Sparse reranking (semantic + lexical)
3. **Memory Checkpoints**: LangGraph with SqliteSaver for state persistence
4. **SQLite Database**: Permanent storage for documents, chat history, and checkpoints
5. **Langfuse Telemetry**: Full tracing and observability

**Next Steps**:
- Configure your Langfuse credentials in `.env`
- Scale retriever to real documents
- Implement actual LLM calls in agent nodes
- Monitor traces in Langfuse dashboard